# Generar Archivos

1. Generar archivos.
2. Subir código, archivos, librería .jar a GCS.
3. Ejecutar job en Dataproc.

In [1]:
!pip install Faker
# !pip install pyspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 16.7 MB/s eta 0:00:00


In [2]:
from datetime import datetime
import random
from faker import Faker
import uuid
from google.cloud import storage

In [3]:
from google.colab import auth
auth.authenticate_user()

In [4]:
project_id = 'western-trilogy-429606-h1'
client = storage.Client(project=project_id)
bucket_name = 'sd-gcolab-bucket-2'
bucket = client.create_bucket(bucket_name)

In [5]:
def crear_campanas(quantity):
    fecha_archivo = datetime.now().strftime('%Y%m%d')
    uuid_4digitos = str(uuid.uuid4().hex)[:4]
    fake = Faker()
    lines = []
    archivo_local = f'campanas_{fecha_archivo}_{uuid_4digitos}.txt'
    blob_name = f'ingested/txt/{archivo_local}'

    for _ in range(quantity):
        nro_campania = fake.random_number(digits=8)
        costo = round(random.uniform(10, 1000), 2)
        pais = fake.country()
        fecha = fake.date_time_between_dates(datetime_start=datetime(2023, 5, 1))
        line = f'{nro_campania}\t{costo}\t{pais}\t{fecha}\n'
        lines.append(line)

    text_content = ''.join(lines)

    with open(archivo_local, 'w') as file:
        file.write(text_content)

    blob = bucket.blob(blob_name)
    blob.upload_from_filename(archivo_local)

    message = f'Se ha generado el archivo de campanias: {blob_name} en el bucket: {bucket_name}'

    return message

archivo_generado = crear_campanas(100000)
print(archivo_generado)


Se ha generado el archivo de campanias: ingested/txt/campanas_20241009_a797.txt en el bucket: sd-gcolab-bucket-2


In [6]:
def crear_archivo_transacciones(quantity):
    fecha_archivo = datetime.now().strftime('%Y%m%d')
    uuid_4digitos = str(uuid.uuid4().hex)[:4]
    fake = Faker()
    lines = []
    archivo_local = f'transacciones_{fecha_archivo}_{uuid_4digitos}.txt'
    blob_name = f'ingested/txt/{archivo_local}'

    for _ in range(quantity):
        nro_transaccion = fake.random_number(digits=8)
        monto = round(random.uniform(10, 10000), 2)
        pais = fake.country()
        fecha = fake.date_time_between_dates(datetime_start=datetime(2023, 5, 1))
        line = f'{nro_transaccion}\t{monto}\t{pais}\t{fecha}\n'
        lines.append(line)

    text_content = ''.join(lines)

    with open(archivo_local, 'w') as file:
        file.write(text_content)

    blob = bucket.blob(blob_name)
    blob.upload_from_filename(archivo_local)

    message = f'Se ha generado el archivo de transacciones: {blob_name} en el bucket: {bucket_name}'

    return message

archivo_generado = crear_archivo_transacciones(100000)
print(archivo_generado)

Se ha generado el archivo de transacciones: ingested/txt/transacciones_20241009_f002.txt en el bucket: sd-gcolab-bucket-2


# Transformaciones

Ejecutar esta porción de código en Dataproc. Prerequisitos:
1. Bucket debe existir.
2. Datasets de BigQuery deben existir: master_layer, raw_layer

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, count, expr

bucket_name = 'sd-gcolab-bucket-2'
project_id = 'western-trilogy-429606-h1'

# Crear la sesión de Spark
spark = SparkSession.builder.appName('txt_to_bq').getOrCreate()
sc = spark.sparkContext
sc.setLogLevel('WARN')
spark.conf.set('temporaryGcsBucket', bucket_name)

files_rdd_cmp = sc.textFile(f'gs://{bucket_name}/ingested/txt/campanas_*.txt')
splitted_rdd_cmp = files_rdd_cmp.map(lambda x: x.split("\t"))
selected_col_rdd_cmp = splitted_rdd_cmp.map(lambda x: (x[0], x[1], x[2], x[3]))
columns_cmp = ["nro_campania","costo","pais","fecha_hora"]
cmp_df = selected_col_rdd_cmp.toDF(columns_cmp)
cmp_df.write.format('bigquery').option('table', project_id+'.raw_layer.r_campanas').mode('append').save()


files_rdd_trx = sc.textFile(f'gs://{bucket_name}/ingested/txt/transacciones_*.txt')
splitted_rdd_trx = files_rdd_trx.map(lambda x: x.split("\t"))
selected_col_rdd_trx = splitted_rdd_trx.map(lambda x: (x[0], x[1], x[2], x[3]))
columns_trx = ["nro_transaccion","monto","pais","fecha_hora"]
trx_df = selected_col_rdd_trx.toDF(columns_trx)
trx_df.write.format('bigquery').option('table', project_id+'.raw_layer.r_transacciones').mode('append').save()

# Leer los datos de neural-truth-406011.raw_layer.r_transacciones
tranx = spark.read.format('bigquery').option('table', project_id+'.raw_layer.r_transacciones').load()

# Calcular fecha y hora
tranx = tranx.withColumn('fecha', tranx.fecha_hora.substr(1, 10))
tranx = tranx.withColumn('hora', tranx.fecha_hora.substr(12, 8))

# Calcular las métricas de transacciones
tranx_agg = tranx.groupBy('pais', 'fecha').agg(
    sum(col('monto').cast('float')).alias('income'),
    count('nro_transaccion').alias('cant_trx'),
    expr('0.00 as costo'),
    expr('0 as cant_cmp')
)

# Leer los datos de neural-truth-406011.raw_layer.r_campanas
campaigns = spark.read.format('bigquery').option('table', project_id+'.raw_layer.r_campanas').load()

# Calcular fecha y hora
campaigns = campaigns.withColumn('fecha', campaigns.fecha_hora.substr(1, 10))
campaigns = campaigns.withColumn('hora', campaigns.fecha_hora.substr(12, 8))

# Calcular las métricas de campañas
campaigns_agg = campaigns.groupBy('pais', 'fecha').agg(
    expr('0.00 as income'),
    expr('0 as cant_trx'),
    sum(col('costo').cast('float')).alias('costo'),
    count('nro_campania').alias('cant_cmp')
)

# Unir las métricas de transacciones y campañas
unionall = tranx_agg.unionAll(campaigns_agg)

# Calcular las métricas agregadas
unionall_agg = unionall.groupBy('pais', 'fecha').agg(
    sum('income').alias('income'),
    sum('cant_trx').alias('cant_trx'),
    sum('costo').alias('cost'),
    sum('cant_cmp').alias('cant_cmp')
)

# Calcular las métricas finales
result = unionall_agg.select(
    col('pais').alias('country'),
    col('cant_trx'),
    col('income'),
    col('cost'),
    (col('income') - col('cost')).alias('revenue'),
    col('cant_cmp'),
    col('fecha').alias('date')
)

# Mostrar el resultado
result.show(5,False)

result.write.format('bigquery').option('table', project_id+'.master_layer.m_performance_country').mode('append').save()

ModuleNotFoundError: No module named 'pyspark'